# CSC8851 HW4 Sample Solution
Reference notebook used by the NeuNet autograder.

In [ ]:
import numpy as np

def relu(x):
    return np.maximum(0, x)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def build_adjacency_matrix(edges, n_nodes):
    adj = np.zeros((n_nodes, n_nodes), dtype=np.float64)
    for i, j in edges:
        adj[i, j] = 1.0
        adj[j, i] = 1.0
    return adj

def gcn_forward(x, adj, weights, biases):
    h = x
    for w, b in zip(weights, biases):
        h = adj @ h @ w + b
        h = relu(h)
    return sigmoid(h)

def neighborhood_sample(adj, target, n_sample, seed=0):
    rng = np.random.default_rng(seed)
    layers = {"output": [target]}
    seen = {target}
    frontier = [target]
    for layer_name in ("hidden2", "hidden1", "input"):
        neighbors = []
        for node in frontier:
            nbrs = np.where(adj[node] > 0)[0].tolist()
            neighbors.extend(nbr for nbr in nbrs if nbr not in seen)
        neighbors = list(dict.fromkeys(neighbors))
        if not neighbors:
            layers[layer_name] = []
            frontier = []
            continue
        k = min(n_sample, len(neighbors))
        chosen = rng.choice(neighbors, size=k, replace=False).tolist()
        layers[layer_name] = chosen
        seen.update(chosen)
        frontier = chosen
    return layers

def softmask(scores, adj):
    masked = np.where(adj > 0, scores, -1e9)
    exp = np.exp(masked - masked.max(axis=-1, keepdims=True))
    denom = exp.sum(axis=-1, keepdims=True)
    denom = np.where(denom == 0, 1.0, denom)
    return exp / denom

def gat_forward(x, adj, weight, attention):
    h = x @ weight
    n_nodes = h.shape[0]
    scores = np.zeros((n_nodes, n_nodes), dtype=np.float64)
    for i in range(n_nodes):
        for j in range(n_nodes):
            pair = np.concatenate([h[i], h[j]])
            scores[i, j] = pair @ attention
    attn = softmask(scores, adj + np.eye(n_nodes))
    return attn @ h